# Arabizi keyboard: Llama 3.2 1B v4-interim (the first synthetic batch; not shipped)

**Name:** this notebook trained the interim run on 15 Nov 2026 under the tag `v4` (run `20261115-030439-llama3.2-1b-lora-v4`); it is kept for the record as `v4-interim` and superseded by v4 (ADR 0038).

One LoRA run, `v4`: the same settings as the shipped `v3` (rank 32, alpha 64, lr 2e-4, 3 epochs) on the v4 export: the 18,039 v3 training rows plus all 25,169 synthetic train rows (agent packs A/B/C and the breaker file, checked by Moustafa), chosen to target new word forms, rarely seen words, new vocabulary and unusual spellings. 94,150 examples, AI-derived share 57% (the 50% cap of ADR 0010 no longer applies: Moustafa's decision, 15 Nov 2026).

**How to run:** upload `arabizi-train-v4` as a **new version** of your `arabizi-train` dataset (or a new dataset). Then Settings → Accelerator → **GPU T4** · **Internet on** · Add Input → that dataset (latest version). Then **Save Version → Save & Run All (Commit)**: about 5½ hours in the background; the outputs stay on the version page.

**Data:** dev and test are never in the export: accuracy is measured on the laptop, with the old dev sets and the new synthetic test set.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as the shipped model


In [ ]:
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# The shipped v3 run: train/runs/20261029-054800-llama3.2-1b-lora-v3.json (rank 32, alpha 64, lr 2e-4, 3 epochs).
RUNS = {
    "v4-interim": ["--rank", "32", "--alpha", "64"],
}

for tag, extra in RUNS.items():
    print(f"\n=== {tag}: {' '.join(extra)}", flush=True)
    # check=True: a failed run stops the notebook instead of exporting nothing.
    subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", "llama3.2-1b", "--data", DATA,
                    "--out", OUT, "--tag", tag, *extra], check=True)
    run = max(glob.glob(f"{OUT}/*-llama3.2-1b-lora-{tag}"))
    subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", f"arabizi-llama3.2-1b-{tag}", run],
                   check=True)
    subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)


## Results and download

The table shows each run's best validation loss (training-style rows: **not** the decision; accuracy on the Mac decides). From the version's **Output** tab, use **Download all** (one zip) and put it in `~/Downloads`; then score the runs on the Mac (docs/TRAINING.md).


In [ ]:
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min)")
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"
